# Machine Learning Assignment 2

### Assignment Objective:

In this assignment You will learn how to use all machine learning algorthms learned so far, this includes: KNN, Decision Tree, Random Forest (Bagging), Boosting (AdaBoost and XGBoost)

The dataset is for predicting lung diseases. The target is the last column "Level" that is a discrete value 'Low', 'Medium', 'High'.

Remember you want to find the best model that is a model where difference between training accuracy and testing accuracy are closest to each other.

## Import libraries:

In [1]:
# Import all libraries needed here:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

## Read your dataset

In [2]:
data = pd.read_csv('/Users/cDub/Downloads/cancer data/cancer patient data sets.csv')
data.head()

,index,Patient Id,Age,Gender,Air Pollution,Alcohol use,Dust Allergy,OccuPational Hazards,Genetic Risk,chronic Lung Disease,...,Fatigue,Weight Loss,Shortness of Breath,Wheezing,Swallowing Difficulty,Clubbing of Finger Nails,Frequent Cold,Dry Cough,Snoring,Level
0,0,P1,33,1,2,4,5,4,3,2,...,3,4,2,2,3,1,2,3,4,Low
1,1,P10,17,1,3,1,5,3,4,2,...,1,3,7,8,6,2,1,7,2,Medium
2,2,P100,35,1,4,5,6,5,5,4,...,8,7,9,2,1,4,6,7,2,High
3,3,P1000,37,1,7,7,7,7,6,7,...,4,2,3,1,4,5,6,7,5,High
4,4,P101,46,1,6,8,7,7,7,6,...,3,2,4,1,4,2,4,2,3,High


## Convert target from Categorical to int code:
The target variable is categorical values "Low", "Medium", "High". Convert the target from categorical to int coding using the LabelEncoder method. The target should be: 0 (means Low), 1 (means Medium), or 2 (means High).

In [3]:
le = LabelEncoder()
data['Level'] = le.fit_transform(data['Level'])
data.head()

,index,Patient Id,Age,Gender,Air Pollution,Alcohol use,Dust Allergy,OccuPational Hazards,Genetic Risk,chronic Lung Disease,...,Fatigue,Weight Loss,Shortness of Breath,Wheezing,Swallowing Difficulty,Clubbing of Finger Nails,Frequent Cold,Dry Cough,Snoring,Level
0,0,P1,33,1,2,4,5,4,3,2,...,3,4,2,2,3,1,2,3,4,1
1,1,P10,17,1,3,1,5,3,4,2,...,1,3,7,8,6,2,1,7,2,2
2,2,P100,35,1,4,5,6,5,5,4,...,8,7,9,2,1,4,6,7,2,0
3,3,P1000,37,1,7,7,7,7,6,7,...,4,2,3,1,4,5,6,7,5,0
4,4,P101,46,1,6,8,7,7,7,6,...,3,2,4,1,4,2,4,2,3,0


## Check for missing values

In [4]:
data.isnull().sum()

index                       0
Patient Id                  0
Age                         0
Gender                      0
Air Pollution               0
Alcohol use                 0
Dust Allergy                0
OccuPational Hazards        0
Genetic Risk                0
chronic Lung Disease        0
Balanced Diet               0
Obesity                     0
Smoking                     0
Passive Smoker              0
Chest Pain                  0
Coughing of Blood           0
Fatigue                     0
Weight Loss                 0
Shortness of Breath         0
Wheezing                    0
Swallowing Difficulty       0
Clubbing of Finger Nails    0
Frequent Cold               0
Dry Cough                   0
Snoring                     0
Level                       0
dtype: int64

## Check for outliers:

In [5]:
# Check for outliers using IQR method

import numpy as np

outlier_counts = {}

# Only numeric columns
numeric_cols = data.select_dtypes(include=np.number).columns

for col in numeric_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ((data[col] < lower_bound) | (data[col] > upper_bound)).sum()

    outlier_counts[col] = outliers

# Display results
outlier_df = pd.DataFrame({
    'Column': outlier_counts.keys(),
    'Outlier Count': outlier_counts.values()
})

outlier_df

,Column,Outlier Count
0,index,0
1,Age,10
2,Gender,0
3,Air Pollution,0
4,Alcohol use,0
5,Dust Allergy,0
6,OccuPational Hazards,0
7,Genetic Risk,0
8,chronic Lung Disease,0
9,Balanced Diet,0


## Data Scaling:

In [6]:
# Separate features before scaling
X = data.drop(columns=['Level','Patient Id','index'])
y = data['Level']

# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame (optional, cleaner output)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled.head()

,Age,Gender,Air Pollution,Alcohol use,Dust Allergy,OccuPational Hazards,Genetic Risk,chronic Lung Disease,Balanced Diet,Obesity,...,Coughing of Blood,Fatigue,Weight Loss,Shortness of Breath,Wheezing,Swallowing Difficulty,Clubbing of Finger Nails,Frequent Cold,Dry Cough,Snoring
0,-0.347848,-0.819903,-0.906679,-0.214954,-0.083340,-0.398718,-0.743202,-1.288162,-1.167040,-0.218941,...,-0.353971,-0.381548,0.065746,-0.980760,-0.870694,-0.328743,-1.224625,-0.838618,-0.418550,0.728655
1,-1.681238,-0.819903,-0.413919,-1.360357,-0.083340,-0.873383,-0.272821,-1.288162,-1.167040,-1.160623,...,-0.766045,-1.273014,-0.387677,1.208436,2.069186,0.993281,-0.805663,-1.384593,1.544171,-0.628245
2,-0.181174,-0.819903,0.078842,0.166847,0.421751,0.075946,0.197560,-0.205673,0.706970,1.193582,...,1.294323,1.847119,1.426018,2.084114,-0.870694,-1.210093,0.032260,1.345283,1.544171,-0.628245
3,-0.014501,-0.819903,1.557123,0.930449,0.926842,1.025275,0.667941,1.418061,1.175473,1.193582,...,1.294323,0.064186,-0.841101,-0.542921,-1.360675,0.111931,0.451222,1.345283,1.544171,1.407105
4,0.735531,-0.819903,1.064362,1.312250,0.926842,1.025275,1.138323,0.876816,1.175473,1.193582,...,1.706397,-0.381548,-0.841101,-0.105081,-1.360675,0.111931,-0.805663,0.253332,-0.909231,0.050205


## Extract features X and target y from the dataset:

In [7]:
# Extract features (X) and target (y)

X = data.drop(columns=['Level', 'Patient Id', 'index'])
y = data['Level']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 23)
y shape: (1000,)


## Split X and y into X_train, X_test, y_train, y_test

In [8]:
# Split the dataset into training and testing sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,  # IMPORTANT: use scaled data
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Show shapes (good for debugging + grading)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (800, 23)
X_test: (200, 23)
y_train: (800,)
y_test: (200,)


## KNN:
Use KNN and find the best K-neighbor value:

In [9]:
for k in range(1, 16):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    
    train = model.score(X_train, y_train)
    test = model.score(X_test, y_test)
    
    print(k, train, test, abs(train-test))

1 1.0 1.0 0.0
2 1.0 1.0 0.0
3 1.0 1.0 0.0
4 1.0 1.0 0.0
5 1.0 1.0 0.0
6 1.0 1.0 0.0
7 1.0 1.0 0.0
8 1.0 1.0 0.0
9 1.0 1.0 0.0
10 1.0 1.0 0.0
11 1.0 1.0 0.0
12 0.9925 0.98 0.012500000000000067
13 0.9925 0.98 0.012500000000000067
14 0.9925 0.98 0.012500000000000067
15 0.9925 0.98 0.012500000000000067


## Decision Tree
Use decision tree and find the best max depth value:

In [10]:
for d in range(1, 16):
    model = DecisionTreeClassifier(max_depth=d)
    model.fit(X_train, y_train)
    
    train = model.score(X_train, y_train)
    test = model.score(X_test, y_test)
    
    print(d, train, test, abs(train-test))

1 0.64625 0.65 0.003750000000000031
2 0.885 0.905 0.020000000000000018
3 0.95875 0.965 0.006249999999999978
4 0.99 0.99 0.0
5 1.0 1.0 0.0
6 1.0 1.0 0.0
7 1.0 1.0 0.0
8 1.0 1.0 0.0
9 1.0 1.0 0.0
10 1.0 1.0 0.0
11 1.0 1.0 0.0
12 1.0 1.0 0.0
13 1.0 1.0 0.0
14 1.0 1.0 0.0
15 1.0 1.0 0.0


## Random Forest:
Use random forest to find the best number of estimators and max depth:

In [11]:
for n in [10,50,100]:
    for d in [3,5,10,None]:
        model = RandomForestClassifier(n_estimators=n, max_depth=d)
        model.fit(X_train, y_train)
        
        train = model.score(X_train, y_train)
        test = model.score(X_test, y_test)
        
        print(n, d, train, test, abs(train-test))

10 3 1.0 1.0 0.0
10 5 1.0 1.0 0.0
10 10 1.0 1.0 0.0
10 None 1.0 1.0 0.0
50 3 0.9775 0.99 0.012499999999999956
50 5 1.0 1.0 0.0
50 10 1.0 1.0 0.0
50 None 1.0 1.0 0.0
100 3 0.99 0.99 0.0
100 5 1.0 1.0 0.0
100 10 1.0 1.0 0.0
100 None 1.0 1.0 0.0


## AdaBoost
Use AdaBoost with none in the estimator parameter to find the best value for number of estimators. Use learning_rate = 0.01, Check the website: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html


In [12]:
for n in [10,50,100,150]:
    model = AdaBoostClassifier(n_estimators=n, learning_rate=0.01)
    model.fit(X_train, y_train)
    
    train = model.score(X_train, y_train)
    test = model.score(X_test, y_test)
    
    print(n, train, test, abs(train-test))

10 0.64625 0.65 0.003750000000000031
50 0.65875 0.65 0.008749999999999925
100 0.65875 0.65 0.008749999999999925
150 0.65875 0.65 0.008749999999999925


## XGBoost:
Use the slides that uses XGBoost. To install XGBoost, use: pip install xgboost



In [13]:
try:
    from xgboost import XGBClassifier

    model = XGBClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        objective='multi:softmax',
        num_class=3
    )

    model.fit(X_train, y_train)

    print("Train:", model.score(X_train, y_train))
    print("Test:", model.score(X_test, y_test))

except:
    print("XGBoost not installed — skipping")

Train: 1.0
Test: 1.0
